# 🏗️ Buyer Segmentation and Investment Profiling

## 📖 Notebook 03 — Feature Engineering

### Objective

This notebook creates business-oriented features required for buyer segmentation and investment profiling.

### Planned Features

- Buyer Age
- Total Properties Purchased
- Total Investment
- Average Property Price
- Average Property Area
- Preferred Property Category
- Active Listings
- Buyer Investment Profile

The final output will be a client-level analytical dataset.

In [2]:
# ============================================
# Import Libraries
# ============================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
# ============================================
# Load Cleaned Datasets
# ============================================

clients_df = pd.read_csv("../data/processed/clients_cleaned.csv")

properties_df = pd.read_csv("../data/processed/properties_cleaned.csv")

print("Cleaned datasets loaded successfully.")

Cleaned datasets loaded successfully.


In [4]:
print("Clients Shape:", clients_df.shape)
print("Properties Shape:", properties_df.shape)

Clients Shape: (2000, 12)
Properties Shape: (10000, 9)


In [5]:
# ============================================
# Convert date_of_birth to datetime
# ============================================

clients_df["date_of_birth"] = pd.to_datetime(
    clients_df["date_of_birth"]
)

In [6]:
# ============================================
# Create Age Feature
# ============================================

today = pd.Timestamp.today()

clients_df["age"] = (
    (today - clients_df["date_of_birth"]).dt.days // 365
)

print("Age feature created successfully.")

Age feature created successfully.


In [7]:
clients_df[["date_of_birth", "age"]].head(10)

,date_of_birth,age
0,1968-05-11,58
1,1962-11-26,63
2,1959-04-07,67
3,1959-11-25,66
4,1976-02-28,50
5,1957-03-06,69
6,1947-05-24,79
7,1969-10-17,56
8,1975-10-05,50
9,1966-06-17,60


In [8]:
clients_df["age"].describe()

count    2000.00000
mean       55.65750
std        17.36891
min        25.00000
25%        41.00000
50%        56.00000
75%        70.00000
max        95.00000
Name: age, dtype: float64

In [9]:
print("Minimum Age :", clients_df["age"].min())
print("Maximum Age :", clients_df["age"].max())

Minimum Age : 25
Maximum Age : 95


In [10]:
# ============================================
# Total Properties Purchased by Each Client
# ============================================

property_count = (
    properties_df
    .groupby("client_ref")
    .size()
    .reset_index(name="total_properties")
)

property_count.head()

,client_ref,total_properties
0,C0001,4
1,C0002,5
2,C0003,5
3,C0004,6
4,C0005,13


In [11]:
# ============================================
# Merge Property Count
# ============================================

clients_df = clients_df.merge(
    property_count,
    left_on="client_id",
    right_on="client_ref",
    how="left"
)

clients_df.head()

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age,client_ref,total_properties
0,C0001,Individual,Kareem,Liu,1968-05-11,F,USA,California,Home,4,Yes,Website,58,C0001,4
1,C0002,Individual,Trystan,Oconnor,1962-11-26,M,USA,California,Home,1,No,Website,63,C0002,5
2,C0003,Individual,Kale,Gay,1959-04-07,M,USA,California,Home,4,Yes,Agency,67,C0003,5
3,C0004,Individual,Russell,Gross,1959-11-25,M,USA,California,Home,5,No,Website,66,C0004,6
4,C0005,Company,Marleez,Co,1976-02-28,M,USA,California,Investment,5,No,Website,50,C0005,13


In [12]:
clients_df["total_properties"] = (
    clients_df["total_properties"]
    .fillna(0)
    .astype(int)
)

In [13]:
clients_df.drop(columns="client_ref", inplace=True)

In [14]:
clients_df[["client_id", "total_properties"]].head(10)

,client_id,total_properties
0,C0001,4
1,C0002,5
2,C0003,5
3,C0004,6
4,C0005,13
5,C0006,5
6,C0007,11
7,C0008,5
8,C0009,5
9,C0010,5


In [15]:
clients_df["total_properties"].describe()

count    2000.0000
mean        3.6525
std         0.8397
min         3.0000
25%         3.0000
50%         4.0000
75%         4.0000
max        13.0000
Name: total_properties, dtype: float64

In [16]:
# ============================================
# Total Investment by Each Client
# ============================================

total_investment = (
    properties_df
    .groupby("client_ref")["sale_price"]
    .sum()
    .reset_index(name="total_investment")
)

total_investment.head()

,client_ref,total_investment
0,C0001,1246764.72
1,C0002,1841095.93
2,C0003,1661457.59
3,C0004,1608263.51
4,C0005,3653385.38


In [17]:
# ============================================
# Merge Total Investment
# ============================================

clients_df = clients_df.merge(
    total_investment,
    left_on="client_id",
    right_on="client_ref",
    how="left"
)

In [18]:
clients_df["total_investment"] = (
    clients_df["total_investment"]
    .fillna(0)
)

In [19]:
clients_df.drop(columns="client_ref", inplace=True)

In [20]:
clients_df[
    ["client_id", "total_properties", "total_investment"]
].head(10)


,client_id,total_properties,total_investment
0,C0001,4,1246764.72
1,C0002,5,1841095.93
2,C0003,5,1661457.59
3,C0004,6,1608263.51
4,C0005,13,3653385.38
5,C0006,5,1514131.06
6,C0007,11,3251147.83
7,C0008,5,1476730.41
8,C0009,5,1820832.35
9,C0010,5,1732434.00


In [21]:
clients_df["total_investment"].describe()

count    2.000000e+03
mean     1.260375e+06
std      3.478307e+05
min      4.636120e+05
25%      1.025238e+06
50%      1.220893e+06
75%      1.441974e+06
max      3.653385e+06
Name: total_investment, dtype: float64

In [22]:
# ============================================
# Average Property Price Per Client
# ============================================

avg_property_price = (
    properties_df
    .groupby("client_ref")["sale_price"]
    .mean()
    .reset_index(name="avg_property_price")
)

avg_property_price.head()

,client_ref,avg_property_price
0,C0001,311691.180000
1,C0002,368219.186000
2,C0003,332291.518000
3,C0004,268043.918333
4,C0005,281029.644615


In [23]:
# ============================================
# Merge Average Property Price
# ============================================

clients_df = clients_df.merge(
    avg_property_price,
    left_on="client_id",
    right_on="client_ref",
    how="left"
)

In [24]:
clients_df["avg_property_price"] = (
    clients_df["avg_property_price"]
    .fillna(0)
)

In [25]:
clients_df.drop(columns="client_ref", inplace=True)

In [26]:
clients_df[
    [
        "client_id",
        "total_properties",
        "total_investment",
        "avg_property_price"
    ]
].head()

,client_id,total_properties,total_investment,avg_property_price
0,C0001,4,1246764.72,311691.180000
1,C0002,5,1841095.93,368219.186000
2,C0003,5,1661457.59,332291.518000
3,C0004,6,1608263.51,268043.918333
4,C0005,13,3653385.38,281029.644615


In [27]:
clients_df["avg_property_price"].describe()

count      2000.000000
mean     347089.955869
std       69721.821814
min      154537.316667
25%      295807.677708
50%      341523.383333
75%      390843.316875
max      563423.500000
Name: avg_property_price, dtype: float64

In [28]:
# ============================================
# Average Property Area Per Client
# ============================================

avg_property_area = (
    properties_df
    .groupby("client_ref")["floor_area_sqft"]
    .mean()
    .reset_index(name="avg_property_area")
)

avg_property_area.head()

,client_ref,avg_property_area
0,C0001,983.885000
1,C0002,1187.942000
2,C0003,1058.110000
3,C0004,937.103333
4,C0005,927.296154


In [29]:
clients_df = clients_df.merge(
    avg_property_area,
    left_on="client_id",
    right_on="client_ref",
    how="left"
)

In [30]:
clients_df["avg_property_area"] = (
    clients_df["avg_property_area"]
    .fillna(0)
)

In [31]:
clients_df.drop(columns="client_ref", inplace=True)

In [32]:
clients_df[
    [
        "client_id",
        "avg_property_area"
    ]
].head()

,client_id,avg_property_area
0,C0001,983.885000
1,C0002,1187.942000
2,C0003,1058.110000
3,C0004,937.103333
4,C0005,927.296154


In [33]:
clients_df.to_csv(
    "../data/processed/final_buyer_dataset.csv",
    index=False
)

print("Final buyer dataset saved successfully.")

Final buyer dataset saved successfully.
